# Organization

## Init

In [0]:
from pyspark.sql import functions as F

## Loading Data From Silver Schema

In [0]:
pitch_df = spark.table('pitch_data_2026.silver.pitch_data')

## Final Clean

### Changing Column Names

In [0]:
rename_cols = {
    'pitcher': 'pitcher_id',
    'pitch_type': 'pitch_abbr',
    'game_pk': 'game_id',
    'api_break_z_with_gravity': 'break_z',
    'api_break_x_arm': 'break_x'
}

for old_col, new_col in rename_cols.items():
    pitch_df = pitch_df.withColumnRenamed(old_col, new_col)

### Ording Columns

In [0]:
pitch_df = pitch_df.select(
    'game_date', 'game_id', 'pitcher_id', 'player_name', 'age_pit_legacy', 'p_throws', 'pitch_abbr', 'pitch_name',
    'inning', 'inning_topbot', 'balls', 'strikes', 'events', 'description', 'total_pitch_count', 'zone', 
    'attack_zone', 'release_speed', 'release_spin_rate', 'pfx_x', 'pfx_z', 'break_x', 'break_z','plate_x', 'plate_z'
)

## Writing into Gold Schema

### Player Identification Data Frame

In [0]:
pitch_df.createOrReplaceTempView('pitch_df_sql')

player_id = spark.sql(
    """
    SELECT DISTINCT pitcher_id, player_name,
    age_pit_legacy FROM pitch_df_sql
    ORDER BY pitcher_id
    """
)

# Handedness was excluded because of the possibility of potential switch pitchers.

pitch_df = pitch_df.drop('player_name', 'age_pit_legacy')


### Seperating By Month

In [0]:
months = {
    'march':'03',
    'april':'04',
    'may':'05',
    'june':'06',
    'july':'07',
    'august':'08',
    'september':'09'
}

for m,d in months.items():
    globals()[f'{m}_2026_pitches'] = pitch_df.filter(F.month("game_date") == d)

### Writing into Gold Schema

In [0]:
# Player df
player_id.write.mode("overwrite").saveAsTable("pitch_data_2026.gold.player_identification")

# Monthly dfs
month_dfs = {
    'march': march_2026_pitches,
    'april': april_2026_pitches,
    'may': may_2026_pitches,
    'june': june_2026_pitches,
    'july': july_2026_pitches,
    'august': august_2026_pitches,
    'september': september_2026_pitches
}

for m,df in month_dfs.items():
    df.write.mode("overwrite").saveAsTable(f"pitch_data_2026.gold.{m}_2026_pitches")